# Bước 0: predictor còn có thể cải thiện bao nhiêu? (oracle trên một nửa tập val)

Với mỗi clip val, probe của Meta được chấm với 4 loại "token tương lai" khác nhau:

| biến thể | token tương lai đưa vào probe |
|---|---|
| `base` | do **predictor của V-JEPA 2** dự đoán (như bản phát hành) |
| `oracle` | token **thật** mà encoder trích từ chính thời điểm được dự đoán (giống một predictor hoàn hảo) |
| `enconly` | không có token tương lai nào |
| `copylast` | lặp lại bước thời gian cuối cùng ("không có gì thay đổi") |

Kèm theo là cosine giữa token dự đoán và token thật. Notebook tính cho cả giao thức chuẩn (`action_start`) và giao thức gốc
(`official`), có sửa frame id ở 5 video không phải 60fps. Để tiết kiệm thời gian tải, chỉ dùng **43 video có mật độ clip cao nhất**,
chứa khoảng 50% số clip và nặng khoảng 60 GB.

**Cách đọc kết quả:** nếu `oracle − base` lớn (vài điểm trở lên), một predictor tốt hơn sẽ giúp nhiều, nên đáng pretrain lại predictor.
Nếu nhỏ, nên đổi hướng.

**Cài đặt:** GPU T4 x2, Internet On. Mất khoảng 4–6 giờ (chủ yếu là tải video). Nếu bị ngắt giữa chừng: *Add Input* output
của lần chạy trước, rồi chạy lại.

In [ ]:
BRANCH = "claude/vjepa2-verify-improve-dsfgtg"
VJEPA2_COMMIT = "204698b45b3712590f06245fbfba32d3be539812"  # version the code was checked against
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /kaggle/working/repo
!git clone -q https://github.com/facebookresearch/vjepa2.git /kaggle/working/vjepa2 && cd /kaggle/working/vjepa2 && git checkout -q $VJEPA2_COMMIT
!pip install -q decord webdataset timm einops
import os
os.environ["VJEPA2_ROOT"] = "/kaggle/working/vjepa2"
R = "/kaggle/working/repo"
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq install -y aria2 > /dev/null 2>&1 && aria2c --version | head -1
!apt-get -qq install -y aria2 > /dev/null 2>&1 && pip install -q av

In [ ]:
# Checkpoints (~1.3GB encoder+predictor, probe) -> /tmp so they do not count toward the 20GB output limit
!mkdir -p /tmp/ckpt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/vitl.pt
!wget -q -nc -P /tmp/ckpt https://dl.fbaipublicfiles.com/vjepa2/evals/ek100-vitl-256.pt
!ls -lh /tmp/ckpt

In [ ]:
# Official annotations (the full train csv is required: probe output classes are derived from it)
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /kaggle/working/ek100-ann
!ls /kaggle/input; find /kaggle/input -maxdepth 3 | head -30

In [ ]:
import glob, os, shutil
OUT = "/kaggle/working/logits_oracle"
prev = glob.glob("/kaggle/input/**/logits_oracle", recursive=True)
if prev:
    shutil.copytree(prev[0], OUT, dirs_exist_ok=True); print("resumed from", prev[0])
!python $R/scripts/prepare_ek100.py --download_layout --video_root /kaggle/tmp/ek100 \
    --train_csv /kaggle/working/ek100-ann/EPIC_100_train.csv \
    --val_csv /kaggle/working/ek100-ann/EPIC_100_validation.csv \
    --video_info /kaggle/working/ek100-ann/EPIC_100_video_info.csv --out_dir /kaggle/working/work | grep '"val_clips"'

In [ ]:
# Subset: videos with the most clips per second of video, until 50% of the val clips are covered
import pandas as pd
clips = pd.read_csv("/kaggle/working/work/val_clips.csv")
vi = pd.read_csv("/kaggle/working/ek100-ann/EPIC_100_video_info.csv").set_index("video_id")
g = clips.groupby("video_id").size().to_frame("n")
g["dur"] = vi.loc[g.index, "duration"]
g = g.assign(dens=g.n / g.dur).sort_values("dens", ascending=False)
g["cum"] = g.n.cumsum() / g.n.sum()
subset = list(g.index[: (g.cum < 0.5).sum() + 1])
VIDEOS = "/kaggle/working/oracle_videos.txt"
open(VIDEOS, "w").write("\n".join(subset))
print(len(subset), "videos,", int(g.n[subset].sum()), "clips,", f"{g.dur[subset].sum() / 3600:.1f} h of video")

In [ ]:
!python $R/scripts/stream_eval_ek100.py --work_dir /kaggle/working/work \
    --video_info /kaggle/working/ek100-ann/EPIC_100_video_info.csv --out_dir $OUT --videos_file $VIDEOS \
    --vitl_ckpt /tmp/ckpt/vitl.pt --probe_ckpt /tmp/ckpt/ek100-vitl-256.pt \
    --gpus 0 1 --anchors action_start official --dtype fp16 --eval_args "--oracle --fix_frame_ids" \
    --parallel_downloads 3 --connections 8 --max_videos_on_disk 10 --max_hours 10.5

In [ ]:
for anchor in ("action_start", "official"):
    print(f"\n========== {anchor} ==========")
    !python $R/scripts/summarize_oracle.py --work_dir /kaggle/working/work --logits_root $OUT --anchor {anchor} \
        --only_videos $VIDEOS --out /kaggle/working/oracle_{anchor}.json

Gửi lại output của cell cuối (2 bảng) để quyết định có nên pretrain lại predictor không.